# Model building LLM

In [21]:
# installations

%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# importing
import os, time, math, json, requests
from pathlib import Path
from datetime import datetime, timedelta, timezone
import pandas as pd, numpy as np
from dotenv import load_dotenv

In [88]:
# load .env and read the key
load_dotenv(dotenv_path=Path(".env"), override=False)   # keeps OS env if already set
TM_API_KEY = os.getenv("TICKETMASTER_API_KEY")
assert TM_API_KEY, "TICKETMASTER_API_KEY is missing. Put it in .env or your shell env."
print("Ticketmaster key loaded:", TM_API_KEY[:4] + "***")  # mask for safety

Ticketmaster key loaded: wCTa***


In [ ]:
# Constants
EVENTS_JSON_PATH = "events.json"

START_ISO = datetime.now(timezone.utc).strftime("%Y-%m-%dT00:00:00Z")
END_ISO   = (datetime.now(timezone.utc)+timedelta(days=7)).strftime("%Y-%m-%dT23:59:59Z")

probe = requests.get(
    "https://app.ticketmaster.com/discovery/v2/events.json",
    params={"apikey": TM_API_KEY, "countryCode": "CA", "size": 1,
            "startDateTime": START_ISO, "endDateTime": END_ISO},
    timeout=20
)
print("Status:", probe.status_code)
print(probe.text[:300])

Status: 200
{"_embedded":{"events":[{"name":"Calgary Flames vs. Chicago Blackhawks","type":"event","id":"1778vxG62h3pgvP","test":false,"url":"https://www.ticketmaster.ca/calgary-flames-vs-chicago-blackhawks-calgary-alberta-11-07-2025/event/110062F6D25B5065","locale":"en-us","images":[{"ratio":"3_2","url":"https


In [ ]:
def fetch_page(city, page):
    params = {
        "apikey": TM_API_KEY,
        "city": city,
        "countryCode": "CA",
        "size": 200,
        "page": page,
        "sort": "date,asc",
        "startDateTime": START_ISO,
        "endDateTime": END_ISO,
    }
    r = requests.get(f"https://app.ticketmaster.com/discovery/v2/{EVENTS_JSON_PATH}", params=params, timeout=30)
    r.raise_for_status()
    return r.json()

In [ ]:
# tiny helpers
def tm_url(path=EVENTS_JSON_PATH): 
    return f"https://app.ticketmaster.com/discovery/v2/{path}"

def clean_float(x): 
    try: return float(x)
    except: return np.nan

def get_price_range(ev):  # extract min/max/currency
    pr = ev.get("priceRanges") or []
    if pr:
        pr0 = pr[0]
        return clean_float(pr0.get("min")), clean_float(pr0.get("max")), pr0.get("currency")
    return np.nan, np.nan, None

def get_venue(ev):  # extract first venue
    vlist = (ev.get("_embedded") or {}).get("venues") or []
    if not vlist: return {}, None
    v = vlist[0]
    city = ((v.get("city") or {}).get("name") or "") 
    state = ((v.get("state") or {}).get("stateCode") or (v.get("state") or {}).get("name") or "")
    country = ((v.get("country") or {}).get("countryCode") or (v.get("country") or {}).get("name") or "")
    return {
        "venue": v.get("name") or "",
        "city": city,
        "state": state,
        "country": country,
        "lat": (v.get("location") or {}).get("latitude"),
        "lon": (v.get("location") or {}).get("longitude"),
    }, v

In [ ]:
# fetch one page
def fetch_page(city, page):
    params = {
        "apikey": TM_API_KEY,
        "city": city,
        "countryCode": COUNTRY_CODE,
        "size": SIZE,
        "page": page,
        "sort": "date,asc",
        "startDateTime": START_ISO,
        "endDateTime": END_ISO,
    }
    r = requests.get(tm_url(EVENTS_JSON_PATH), params=params, timeout=30)
    if r.status_code == 429:  # rate limited
        time.sleep(1.5); r = requests.get(tm_url(EVENTS_JSON_PATH), params=params, timeout=30)
    r.raise_for_status()
    return r.json()

In [ ]:
# Helper functions for fetch_city
def _fetch_first_page(city):
    """Fetch the first page for a city."""
    try:
        return fetch_page(city, 0)
    except Exception as e:
        print(f"[{city}] error: {e}")
        return None

def _get_total_pages(first_page):
    """Extract total pages from first page response."""
    page_info = first_page.get("page") or {}
    return page_info.get("totalPages", 1)

def _fetch_remaining_pages(city, total_pages):
    """Fetch remaining pages for a city."""
    payloads = []
    for p in range(1, total_pages):
        time.sleep(SLEEP_SEC)
        try:
            payloads.append(fetch_page(city, p))
        except Exception as e:
            print(f"[{city}] page {p} error: {e}")
            break
    return payloads

def _extract_event_dates(ev):
    """Extract date information from event."""
    dates = ev.get("dates") or {}
    start = dates.get("start") or {}
    return {
        "local_date": start.get("localDate") or "",
        "local_time": start.get("localTime") or "",
        "start_iso": start.get("dateTime") or ""
    }

def _extract_event_classifications(ev):
    """Extract classification information from event."""
    classifications = ev.get("classifications") or [{}]
    first_class = classifications[0] if classifications else {}
    segment = (first_class.get("segment") or {}).get("name") or ""
    genre = (first_class.get("genre") or {}).get("name") or ""
    subgenre = (first_class.get("subGenre") or {}).get("name") or ""
    return {"segment": segment, "genre": genre, "subgenre": subgenre}

def _process_event_to_row(ev):
    """Convert a single event to a row dictionary."""
    price_min, price_max, currency = get_price_range(ev)
    venue_dict, _ = get_venue(ev)
    dates = _extract_event_dates(ev)
    classifications = _extract_event_classifications(ev)
    
    return {
        "event_id": ev.get("id"),
        "title": ev.get("name") or "",
        "url": ev.get("url") or "",
        "start_dt_iso": dates["start_iso"],
        "start_date": dates["local_date"],
        "start_time": dates["local_time"],
        "segment": classifications["segment"],
        "genre": classifications["genre"],
        "subgenre": classifications["subgenre"],
        "price_min": price_min,
        "price_max": price_max,
        "currency": currency,
        **venue_dict,
        "source": "ticketmaster",
        "raw_market": ev.get("marketId") or "",
    }

def _process_payloads_to_rows(payloads):
    """Process all payloads and extract event rows."""
    rows = []
    for js in payloads:
        events = (js.get("_embedded") or {}).get("events") or []
        for ev in events:
            rows.append(_process_event_to_row(ev))
    return rows

# fetch all pages for a city
def fetch_city(city):
    first = _fetch_first_page(city)
    if first is None:
        return pd.DataFrame()
    
    total_pages = _get_total_pages(first)
    remaining_payloads = _fetch_remaining_pages(city, total_pages)
    payloads = [first] + remaining_payloads
    
    rows = _process_payloads_to_rows(payloads)
    return pd.DataFrame(rows)

In [95]:
# run fetch for all cities
frames = []
for c in CITIES:
    print(f"Fetching {c}...")
    df_city = fetch_city(c)
    print(f"  -> {len(df_city)} rows")
    frames.append(df_city)
tm_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print("Total rows:", len(tm_df))
tm_df.head(5)

Fetching Toronto...
  -> 283 rows
Fetching Montreal...
  -> 39 rows
Fetching Vancouver...
  -> 56 rows
Fetching Calgary...
  -> 20 rows
Fetching Ottawa...
  -> 20 rows
Total rows: 418


,event_id,title,url,start_dt_iso,start_date,start_time,segment,genre,subgenre,price_min,price_max,currency,venue,city,state,country,lat,lon,source,raw_market
0,rZ7HnEZ1Af43PN,TMU Bold Men's Hockey - 2025-26 Season Pass,https://www.ticketweb.ca/event/tmu-bold-mens-h...,2025-10-02T22:15:00Z,2025-10-02,18:15:00,Sports,Hockey,,110.0,110.0,CAD,Mattamy Athletic Centre,Toronto,ON,CA,43.662302,-79.379993,ticketmaster,
1,rZ7HnEZ1Af44CK,TMU Bold Women's Volleyball - 2025-26 Season Pass,https://www.ticketweb.ca/event/tmu-bold-womens...,2025-10-23T22:00:00Z,2025-10-23,18:00:00,Sports,Volleyball,Volleyball,110.0,110.0,CAD,Mattamy Athletic Centre,Toronto,ON,CA,43.662302,-79.379993,ticketmaster,
2,rZ7HnEZ1Af44GN,TMU Bold Men's Volleyball - 2025-26 Season Pass,https://www.ticketweb.ca/event/tmu-bold-mens-v...,2025-10-23T22:00:00Z,2025-10-23,18:00:00,Sports,Volleyball,Volleyball,110.0,110.0,CAD,Mattamy Athletic Centre,Toronto,ON,CA,43.662302,-79.379993,ticketmaster,
3,rZ7HnEZ1Af430N,TMU Bold Women's Hockey - 2025-26 Season Pass,https://www.ticketweb.ca/event/tmu-bold-womens...,2025-10-24T16:00:00Z,2025-10-24,12:00:00,Sports,Hockey,,110.0,110.0,CAD,Mattamy Athletic Centre,Toronto,ON,CA,43.662302,-79.379993,ticketmaster,
4,rZ7HnEZ1Af4f_K,TMU Bold Women's Basketball - 2025-26 Season Pass,https://www.ticketweb.ca/event/tmu-bold-womens...,2025-10-31T16:00:00Z,2025-10-31,12:00:00,Sports,Basketball,,110.0,110.0,CAD,Mattamy Athletic Centre,Toronto,ON,CA,43.662302,-79.379993,ticketmaster,


In [96]:
# clean + unify datetimes
tm_df["start_ts"] = pd.to_datetime(tm_df["start_dt_iso"], errors="coerce")
tm_df["price_use"] = tm_df["price_min"].fillna(tm_df["price_max"])
tm_df = tm_df.dropna(subset=["title"]).reset_index(drop=True)

In [97]:
# dedupe
dedupe_keys = [k for k in ["event_id","title","city","start_ts"] if k in tm_df.columns]
if dedupe_keys:
    tm_df = (tm_df.sort_values(["start_ts","price_use"])
                  .drop_duplicates(dedupe_keys, keep="first")
                  .reset_index(drop=True))
print("After dedupe:", len(tm_df))

After dedupe: 418


In [98]:
# save artifacts
tm_df.to_parquet(OUT_DIR/"ticketmaster_events.parquet", index=False)
tm_df.to_csv(OUT_DIR/"ticketmaster_events.csv", index=False)
print("Saved:", (OUT_DIR/"ticketmaster_events.csv").resolve())

Saved: D:\LOYALIST_COLLEGE\Semester_4\AIP\VoyagerAI\model\clean_outputs\ticketmaster_events.csv
